# Speech Emotion Recognition (RAVDESS) - corrected pipeline

**Replaces `jugantarSER.ipynb`.** Same model architecture, corrected experimental protocol.

### What changed and why

| # | Original fault | Fix in this notebook |
|---|---|---|
| S1 | Each `.wav` produced a clean **and** a noise-augmented spectrogram, both appended to `X_list` **before** `train_test_split(...)`. ~80% of validation samples had a near-identical twin in training. | Split by **actor first**, extract features second, augment **training actors only**. |
| S2 | All 24 actors appeared in both train and validation, so speaker identity was learnable. | Actor-disjoint partition, asserted at runtime. |
| S3 | RAVDESS records 2 takes of each line; a random split separated them. | Eliminated as a side effect of actor-disjoint splitting. |
| S4 | 4-class Neutral score used `max(p_neutral, p_calm)`. The deployed system **sums** related probabilities. | Summation over all eight classes, matching deployment. |
| S5 | `mask = y_true <= 4` discarded fearful/disgust/surprised, leaving 340 of 576 samples. | All eight classes retained and folded into four. |

**Expect the headline number to fall to roughly 65-75% for four-class.** That drop is the
measurement of how much the old protocol was inflating. It is the defensible number.

### Splits

| Split | Actors | Purpose |
|---|---|---|
| train | 1-16 (8M / 8F) | fitting |
| val | 17-20 (2M / 2F) | early stopping and checkpoint selection |
| test | 21-24 (2M / 2F) | **reported result only** |

RAVDESS actors alternate male (odd) / female (even), so every subset is gender balanced.

## 1. Data

In [ ]:
!pip install -q kaggle
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d uwrfkaggler/ravdess-emotional-speech-audio
!unzip -q -o ravdess-emotional-speech-audio.zip -d ravdess_data

In [ ]:
import os, gc, json, warnings
import numpy as np
import librosa, cv2
import tensorflow as tf
from tensorflow.keras.utils import to_categorical
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings('ignore')

DATASET_PATH = 'ravdess_data/audio_speech_actors_01-24/'
SEED = 42

# Actor-disjoint partition. No speaker appears in more than one split.
TRAIN_ACTORS = list(range(1, 17))
VAL_ACTORS   = [17, 18, 19, 20]
TEST_ACTORS  = [21, 22, 23, 24]

EMOTION8 = ['neutral', 'calm', 'happy', 'sad',
            'angry', 'fearful', 'disgust', 'surprised']

# Deployed 8 -> 4 mapping by SUMMATION. Every class is used.
GROUPS_4 = {0: [0, 1],   # Neutral = neutral + calm
            1: [2, 7],   # Happy   = happy   + surprised
            2: [3, 5],   # Sad     = sad     + fearful
            3: [4, 6]}   # Angry   = angry   + disgust
CLASS4 = ['Neutral', 'Happy', 'Sad', 'Angry']
MAP_8_TO_4 = {c8: c4 for c4, g in GROUPS_4.items() for c8 in g}
assert sorted(MAP_8_TO_4) == list(range(8)), 'all eight classes must be mapped'

rng = np.random.default_rng(SEED)
tf.keras.utils.set_random_seed(SEED)
print('TensorFlow', tf.__version__)

## 2. Speaker-independent split  *(replaces original Cell 2)*

The ordering is the entire fix: actors are partitioned **before** any audio is decoded,
so augmented copies can never cross a split boundary.

In [ ]:
def get_spectrogram(y, sr):
    """Unchanged from the original notebook.
    Do NOT add denoise / trim / volume normalisation here - a mismatch between this
    and the Raspberry Pi inference pipeline is what broke the original deployment."""
    melspec = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128)
    melspec_db = librosa.power_to_db(melspec, ref=np.max)
    img = melspec_db - melspec_db.min()
    img = (img / (img.max() if img.max() > 0 else 1.0) * 255).astype(np.uint8)
    img = cv2.resize(img, (224, 224))
    return np.stack((img,) * 3, axis=-1)


def add_noise(data):
    amp = 0.035 * rng.uniform() * np.amax(np.abs(data))
    return data + amp * rng.standard_normal(data.shape[0])


def time_shift(data, max_frac=0.15):
    return np.roll(data, int(rng.uniform(-max_frac, max_frac) * len(data)))


# ---- group files by actor BEFORE touching any audio ----------------------
files_by_split = {'train': [], 'val': [], 'test': []}
for actor_dir in sorted(os.listdir(DATASET_PATH)):
    actor_path = os.path.join(DATASET_PATH, actor_dir)
    if not os.path.isdir(actor_path):
        continue
    for fn in sorted(os.listdir(actor_path)):
        if not fn.endswith('.wav'):
            continue
        actor = int(fn.split('-')[6].split('.')[0])   # filename field 7
        if   actor in TEST_ACTORS:  files_by_split['test'].append((actor_path, fn))
        elif actor in VAL_ACTORS:   files_by_split['val'].append((actor_path, fn))
        elif actor in TRAIN_ACTORS: files_by_split['train'].append((actor_path, fn))

_actors = {k: {int(fn.split('-')[6].split('.')[0]) for _, fn in v}
           for k, v in files_by_split.items()}
assert not (_actors['train'] & _actors['val']),  'actor overlap train/val'
assert not (_actors['train'] & _actors['test']), 'actor overlap train/test'
assert not (_actors['val']   & _actors['test']), 'actor overlap val/test'

print('speaker-independent split verified')
for k in ('train', 'val', 'test'):
    print(f'  {k:5s}: {len(files_by_split[k]):4d} files | actors {sorted(_actors[k])}')

In [ ]:
def build(pairs, augment, name):
    X, y = [], []
    for actor_path, fn in pairs:
        emotion = int(fn.split('-')[2]) - 1           # filename field 3
        audio, sr = librosa.load(os.path.join(actor_path, fn),
                                 duration=3.0, offset=0.5)
        if audio.size == 0:
            continue
        X.append(get_spectrogram(audio, sr)); y.append(emotion)
        if augment:                                    # TRAINING ACTORS ONLY
            X.append(get_spectrogram(add_noise(audio), sr));  y.append(emotion)
            X.append(get_spectrogram(time_shift(audio), sr)); y.append(emotion)
    X = np.asarray(X, dtype=np.uint8)                  # uint8 -> 4x less RAM
    y = np.asarray(y, dtype=np.int64)
    print(f'  {name:5s}: {X.shape[0]:5d} samples  '
          f'{ {EMOTION8[i]: int((y == i).sum()) for i in range(8)} }')
    return X, y


print('extracting features (augmentation on training actors only)...')
Xtr_u8, ytr_int = build(files_by_split['train'], True,  'train')
Xva_u8, yva_int = build(files_by_split['val'],   False, 'val')
Xte_u8, yte_int = build(files_by_split['test'],  False, 'test')

X_train = Xtr_u8.astype(np.float32) / 255.0
X_val   = Xva_u8.astype(np.float32) / 255.0
X_test  = Xte_u8.astype(np.float32) / 255.0
y_train = to_categorical(ytr_int, num_classes=8)
y_val   = to_categorical(yva_int, num_classes=8)
y_test  = to_categorical(yte_int, num_classes=8)

np.savez_compressed('ser_test_set.npz', X=Xte_u8, y=yte_int)   # for quantization later
del Xtr_u8, Xva_u8
gc.collect()

print(f'\nready. train {X_train.shape} | val {X_val.shape} | test {X_test.shape}')

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 3.6), sharey=True)
for a, (name, yy) in zip(ax, [('train', ytr_int), ('val', yva_int), ('test', yte_int)]):
    counts = [int((yy == i).sum()) for i in range(8)]
    sns.barplot(x=EMOTION8, y=counts, ax=a, color='#4d4d4d')
    a.set_title(f'{name}  (n={len(yy)})'); a.tick_params(axis='x', rotation=45)
ax[0].set_ylabel('samples')
plt.suptitle('Class distribution per split (neutral is half-sized in RAVDESS by design)')
plt.tight_layout(); plt.show()

## 3. Model  *(architecture unchanged)*

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (Input, Conv2D, MaxPooling2D,
                                     GlobalAveragePooling2D, Dense, Dropout,
                                     BatchNormalization)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.regularizers import l2
from sklearn.utils.class_weight import compute_class_weight

model = Sequential([
    Input(shape=(224, 224, 3)),
    Conv2D(32, (3, 3), activation='relu'), BatchNormalization(), MaxPooling2D((2, 2)),
    Conv2D(64, (3, 3), activation='relu'), BatchNormalization(), MaxPooling2D((2, 2)),
    Conv2D(128, (3, 3), activation='relu'), BatchNormalization(), MaxPooling2D((2, 2)),
    Conv2D(256, (3, 3), activation='relu'), BatchNormalization(), MaxPooling2D((2, 2)),
    GlobalAveragePooling2D(),
    Dense(128, activation='relu', kernel_regularizer=l2(0.01)),
    Dropout(0.5),
    Dense(8, activation='softmax'),
])
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
              loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

cw = compute_class_weight('balanced', classes=np.arange(8), y=ytr_int)
class_weight = {i: float(w) for i, w in enumerate(cw)}
print('\nclass weights:', {EMOTION8[i]: round(w, 3) for i, w in class_weight.items()})

## 4. Training

Early stopping and checkpointing both watch **validation loss on the val actors**.
`X_test` is not referenced anywhere in this section.

In [ ]:
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=60, batch_size=32, class_weight=class_weight, verbose=2,
    callbacks=[
        EarlyStopping(monitor='val_loss', patience=10,
                      restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4,
                          min_lr=1e-6, verbose=1),
        ModelCheckpoint('jugantarSER_corrected.keras', monitor='val_loss',
                        save_best_only=True, verbose=0),
    ])

best_epoch = int(np.argmin(history.history['val_loss']))
print(f'\nbest epoch (min val_loss): {best_epoch + 1}')
print(f'val accuracy at that epoch : {history.history["val_accuracy"][best_epoch]*100:.2f}%'
      '   <- SELECTION metric, not the reported result')
model.save('jugantarSER_corrected.keras')

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].plot(history.history['accuracy'], label='train')
ax[0].plot(history.history['val_accuracy'], label='val')
ax[0].set_title('Accuracy'); ax[0].set_xlabel('epoch'); ax[0].legend(); ax[0].grid(alpha=.3)
ax[1].plot(history.history['loss'], label='train')
ax[1].plot(history.history['val_loss'], label='val')
ax[1].axvline(best_epoch, ls='--', c='k', lw=1, label='selected')
ax[1].set_title('Loss'); ax[1].set_xlabel('epoch'); ax[1].legend(); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.show()

## 5. Evaluation on held-out actors  *(replaces original Cell 11)*

Three protocols side by side. **B is the one to publish** - it is what the Raspberry Pi
code actually computes. C is the original protocol, included only to measure how much of
the old 94.41% came from the protocol rather than the model.

In [ ]:
from sklearn.metrics import (classification_report, confusion_matrix,
                             accuracy_score, f1_score, balanced_accuracy_score)

prob8 = model.predict(X_test, verbose=0)
y8 = yte_int
pred8 = prob8.argmax(1)

print('=' * 64)
print('A. 8-CLASS, held-out actors')
print('=' * 64)
print(classification_report(y8, pred8, target_names=EMOTION8, digits=3, zero_division=0))
acc8 = accuracy_score(y8, pred8)
print(f'accuracy {acc8*100:.2f}%   '
      f'macro F1 {f1_score(y8, pred8, average="macro", zero_division=0):.3f}')

In [ ]:
# ---- B. 4-class by SUMMATION  <-- THE NUMBER TO PUBLISH -----------------
prob4 = np.stack([prob8[:, idx].sum(axis=1) for _, idx in sorted(GROUPS_4.items())], axis=1)
assert np.allclose(prob4.sum(1), 1.0, atol=1e-4), 'summation must preserve probability mass'

y4 = np.array([MAP_8_TO_4[int(v)] for v in y8])
pred4 = prob4.argmax(1)
acc4 = accuracy_score(y4, pred4)

print('=' * 64)
print('B. 4-CLASS BY SUMMATION (deployed protocol)   <-- REPORT THIS')
print('=' * 64)
print(classification_report(y4, pred4, target_names=CLASS4, digits=3, zero_division=0))
print(f'accuracy          {acc4*100:.2f}%')
print(f'balanced accuracy {balanced_accuracy_score(y4, pred4)*100:.2f}%')
print(f'macro F1          {f1_score(y4, pred4, average="macro", zero_division=0):.3f}')

boot = np.empty(2000)
for b in range(2000):
    i = rng.integers(0, len(y4), len(y4))
    boot[b] = (y4[i] == pred4[i]).mean()
lo, hi = np.quantile(boot, [0.025, 0.975])
print(f'95% CI            [{lo*100:.2f}, {hi*100:.2f}]')
print(f'n samples         {len(y4)}   (all eight classes retained)')

cm = confusion_matrix(y4, pred4)
plt.figure(figsize=(7, 5.5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greys', cbar=False,
            linewidths=1, linecolor='black', xticklabels=CLASS4, yticklabels=CLASS4)
plt.title(f'SER 4-class by summation, held-out actors\naccuracy {acc4*100:.2f}%')
plt.ylabel('True'); plt.xlabel('Predicted')
plt.tight_layout(); plt.show()

In [ ]:
# ---- C. legacy protocol, for the correction table only ------------------
mask = y8 <= 4
p_leg, y_leg = prob8[mask], y8[mask]
true_leg = np.where(y_leg <= 1, 0, y_leg - 1)
pred_leg = np.stack([np.maximum(p_leg[:, 0], p_leg[:, 1]),
                     p_leg[:, 2], p_leg[:, 3], p_leg[:, 4]], axis=1).argmax(1)
acc_leg = accuracy_score(true_leg, pred_leg)

print('=' * 64)
print(f'C. LEGACY max()+filtered protocol (n={int(mask.sum())})  -- comparison only')
print('=' * 64)
print(f'accuracy {acc_leg*100:.2f}%')
print(f'\n>> protocol inflation: {(acc_leg - acc4)*100:+.2f} accuracy points gained by')
print('   discarding three classes and using max() instead of summation.')

In [ ]:
results = {
    'protocol': {
        'split': 'speaker-independent (actor-disjoint)',
        'train_actors': TRAIN_ACTORS, 'val_actors': VAL_ACTORS, 'test_actors': TEST_ACTORS,
        'augmentation': 'training actors only: gaussian noise + time shift',
        'four_class_mapping': 'summation over all eight classes',
        'selection_criterion': 'minimum validation loss on val actors',
    },
    'n_samples': {'train': int(len(ytr_int)), 'val': int(len(yva_int)),
                  'test': int(len(yte_int))},
    'best_epoch': best_epoch + 1,
    'val_accuracy_selection_metric': float(history.history['val_accuracy'][best_epoch]),
    'test_8class_accuracy': float(acc8),
    'test_4class_accuracy_summation_REPORT_THIS': float(acc4),
    'test_4class_macro_f1': float(f1_score(y4, pred4, average='macro', zero_division=0)),
    'test_4class_ci95': [float(lo), float(hi)],
    'test_4class_confusion_matrix': cm.tolist(),
    'legacy_protocol_accuracy': float(acc_leg),
    'protocol_inflation_points': float((acc_leg - acc4) * 100),
    'originally_reported': {'value': 94.41,
                            'why_invalid': ('speaker-dependent random split with augmented '
                                            'twins across the boundary, plus max()-based '
                                            'mapping on a filtered subset')},
}
with open('ser_corrected_results.json', 'w') as f:
    json.dump(results, f, indent=2)
print(json.dumps(results, indent=2))

## 6. What to put in the paper

- **Section 3.2 and the abstract:** replace 94.41% with `test_4class_accuracy_summation_REPORT_THIS`.
- **Methods:** state the split explicitly - "speaker-independent, actors 1-16 / 17-20 / 21-24,
  augmentation applied to training actors only". Reviewers reject unclear protocols far more
  often than modest accuracy.
- **Correction note:** `protocol_inflation_points` is how many accuracy points the original
  evaluation gained from protocol alone. Quote it rather than asserting the correction mattered.
- Keep `jugantarSER_corrected.keras` and `ser_test_set.npz` - the quantization study needs
  both, and it must use this same test set.